# Lun 07/12 · 09:45 — TP v0.12 : pyproject.toml et point d'entrée

Lun 07/12, TP de 10:15 à 11:05. Correction à 11:05, TP fini ou non. La référence v0.12 est publiée à 11:15.

## Objectif

`fleetcheck` devient un outil installable :

- un `pyproject.toml` décrit le package, sa version et ses dépendances ;
- `pip install -e .` crée une commande `fleetcheck`, utilisable depuis n'importe quel dossier ;
- l'option `--config` dit où trouver la configuration quand l'outil tourne hors du dépôt.

## Mode d'emploi du notebook

- Copiez ce notebook à la racine de votre dépôt, à côté du dossier `fleetcheck/`. Le `pyproject.toml` s'écrit à la racine, ouvert dans JupyterLab ; le notebook sert à vérifier chaque étape.
- Le serveur simulé n'est pas nécessaire : les vérifications ne portent que sur l'installation et les options.
- Exécutez d'abord la cellule d'outils ci-dessous. Chaque vérification affiche `OK`, ou `KO` avec ce qui était attendu.
- `%pip install -e .` installe le package dans le venv du noyau, `esgi-python/.venv` : c'est ce venv qui reçoit la commande `fleetcheck`. Les vérifications lancent cette commande depuis un dossier temporaire, hors du dépôt, pour reproduire une installation en CI.
- **Prédire** et **Justifier** marquent les moments clés du TP. Répondez dans la cellule `# Ma réponse :` qui suit, **avant** d'exécuter la suite.
- Rendu en fin de TP : menu *Kernel > Restart Kernel and Run All Cells*, puis commit et push de `pyproject.toml`, `fleetcheck/` et de ce notebook. `fleetcheck.egg-info/`, créé par l'installation, n'est pas commité (`.gitignore`, étape 2).

In [ ]:
# Outils du TP : exécuter cette cellule en premier, sans la modifier.
import os
import subprocess
import sys


def check(obtenu, attendu):
    if obtenu == attendu:
        print("OK")
    elif isinstance(attendu, str) and "\n" in attendu:
        print(f"KO, obtenu :\n{obtenu}\nattendu :\n{attendu}")
    else:
        print(f"KO\n  obtenu  : {obtenu!r}\n  attendu : {attendu!r}")


def check_error(fonction, *args, attendu):
    try:
        fonction(*args)
    except Exception as e:
        check(f"{type(e).__name__}: {e}", attendu)
    else:
        print(f"KO, aucune exception levée. Attendu : {attendu}")


def run(*args):
    """Lance fleetcheck.py comme dans un terminal (vu en v0.8 : subprocess)."""
    r = subprocess.run([sys.executable, "fleetcheck.py", *args], capture_output=True,
                       encoding="utf-8", env={**os.environ, "PYTHONUTF8": "1", "NO_COLOR": "1",
                                              "PYTHONWARNDEFAULTENCODING": "1"})
    print(r.stdout, end="")
    print(r.stderr, end="", file=sys.stderr)
    print(f"[code de sortie : {r.returncode}]")
    if "EncodingWarning" in r.stderr:
        print("KO : un open(), read_text() ou subprocess text=True sans encoding= (ligne indiquée dans l'EncodingWarning ci-dessus)")
    return r


def run_module(*args):
    """Lance le package comme dans un terminal : python -m fleetcheck ..."""
    r = subprocess.run([sys.executable, "-m", "fleetcheck", *args], capture_output=True,
                       encoding="utf-8", env={**os.environ, "PYTHONUTF8": "1", "NO_COLOR": "1",
                                              "PYTHONWARNDEFAULTENCODING": "1"})
    print(r.stdout, end="")
    print(r.stderr, end="", file=sys.stderr)
    print(f"[code de sortie : {r.returncode}]")
    if "EncodingWarning" in r.stderr:
        print("KO : un open(), read_text() ou subprocess text=True sans encoding= (ligne indiquée dans l'EncodingWarning ci-dessus)")
    return r

## Point de départ

Votre package `fleetcheck/` de v0.11, avec `config.json` et les inventaires à la racine.

En retard ou en panne : copiez dans votre dépôt les fichiers de `reference/` au tag `v0.11` du dépôt public.

Vérifiez avant de commencer :

In [ ]:
import importlib.metadata
import os
import sysconfig
import tempfile
import tomllib
from pathlib import Path

r = run_module("--help")
check((r.returncode, r.stdout.startswith("usage: fleetcheck")), (0, True))
check(Path("fleetcheck/cli.py").exists(), True)

## Étapes

| Fin | Étape |
|---|---|
| 10:35 | 1. `pyproject.toml` |
| 10:50 | 2. Point d'entrée et installation éditable |
| 11:05 | 3. Option `--config` |

À 10:40, si plus d'un tiers de la salle n'a pas un `pyproject.toml` lisible, l'enseignant projette celui de la référence et la salle le recopie ; l'étape 3 bascule en devoir.

### 1. `pyproject.toml` — fin 10:35

À la racine du dépôt, trois sections :

- `[build-system]` : `requires = ["setuptools>=61"]`, `build-backend = "setuptools.build_meta"` ;
- `[project]` : `name = "fleetcheck"`, `version = "0.12.0"`, une `description`, `requires-python = ">=3.10"`, `dependencies = ["requests>=2.31"]` ;
- `[tool.setuptools]` : `packages = ["fleetcheck"]`, pour n'empaqueter que ce dossier.

TOML : sections entre crochets, `clé = valeur`, chaînes entre guillemets doubles, listes entre crochets.

Vérification : le fichier se lit avec `tomllib` (bibliothèque standard depuis Python 3.11), ouvert en binaire (`"rb"`) : `tomllib.load` décode lui-même l'UTF-8.

In [ ]:
with open("pyproject.toml", "rb") as f:
    pyproject = tomllib.load(f)
projet = pyproject["project"]
check((projet["name"], projet["version"]), ("fleetcheck", "0.12.0"))
check(any(d.startswith("requests") for d in projet["dependencies"]), True)
check(pyproject["build-system"]["build-backend"], "setuptools.build_meta")

### 2. Point d'entrée et installation éditable — fin 10:50

- Ajoutez la section `[project.scripts]` avec `fleetcheck = "fleetcheck.cli:main"` : à l'installation, pip crée une commande `fleetcheck` qui appelle `main()` du module `fleetcheck.cli`.
- L'installation crée `fleetcheck.egg-info/` à la racine du dépôt : ajoutez `*.egg-info/` au `.gitignore` s'il n'y est pas.
- Installez le package en mode éditable dans le venv du noyau, avec la cellule ci-dessous.

In [ ]:
%pip install -e .

**Justifier** — Pourquoi `pip install -e .` plutôt que `pip install .` pendant le développement ?

In [ ]:
# Ma réponse :

**Prédire** — Après l'installation, on passe `version` à `"0.12.1"` dans `pyproject.toml` sans réinstaller. Que renvoie `importlib.metadata.version("fleetcheck")` ?

In [ ]:
# Ma réponse :

Vérification : la version installée, la commande créée dans le dossier des scripts du venv (`bin/` ou `Scripts\`), et son aide affichée depuis un dossier temporaire.

In [ ]:
check(importlib.metadata.version("fleetcheck"), "0.12.0")
check("*.egg-info/" in Path(".gitignore").read_text(encoding="utf-8").split(), True)
COMMANDE = Path(sysconfig.get_path("scripts")) / ("fleetcheck.exe" if os.name == "nt" else "fleetcheck")
check(COMMANDE.exists(), True)


def fleetcheck(*args, cwd):
    """Lance la commande installée depuis le dossier cwd, comme dans un terminal."""
    r = subprocess.run([str(COMMANDE), *args], cwd=cwd, capture_output=True, encoding="utf-8",
                       env={**os.environ, "PYTHONUTF8": "1", "NO_COLOR": "1"})
    print(r.stdout, end="")
    print(r.stderr, end="", file=sys.stderr)
    print(f"[code de sortie : {r.returncode}]")
    return r


with tempfile.TemporaryDirectory() as ailleurs:
    r = fleetcheck("--help", cwd=ailleurs)
check((r.returncode, r.stdout.startswith("usage: fleetcheck")), (0, True))

Avant l'étape 3 : inventaire de deux lignes pour ses vérifications (`web-1` et `cache-1`, comme en v0.10), écrit par cette cellule fournie :

In [ ]:
depot = Path.cwd()
(depot / "servers-alerte.csv").write_text(
    "name,env,ip,port,role\n"
    "web-1,prod,127.0.0.1,8001,web\n"
    "cache-1,prod,127.0.0.1,8009,cache\n", encoding="utf-8")
check((depot / "servers-alerte.csv").exists(), True)

### 3. Option `--config` — fin 11:05

- Lancée depuis un autre dossier, la commande cherche `config.json` dans ce dossier, pas dans le dépôt. Ajoutez à argparse `--config` (défaut `config.json`) et passez `args.config` à `load_config`.
- `--inventory` existe déjà : avec les deux options, l'appelant dit où sont ses fichiers.

**Prédire** — Depuis un dossier temporaire, `fleetcheck --config <dépôt>/config.json --inventory <dépôt>/servers-alerte.csv` : où s'écrit le rapport ?

In [ ]:
# Ma réponse :

Vérification, cellule fournie : elle lance la commande depuis un dossier temporaire, sans option, puis avec les deux chemins du dépôt.

In [ ]:
with tempfile.TemporaryDirectory() as ailleurs:
    r = fleetcheck(cwd=ailleurs)
    check(((r.stderr.splitlines() or [""])[-1].startswith("ERROR config illisible (config.json) : "), r.returncode), (True, 2))
    r = fleetcheck("--config", str(depot / "config.json"), "--inventory", str(depot / "servers-alerte.csv"), cwd=ailleurs)
    services = [l.split()[1] for l in r.stdout.splitlines() if l.startswith(("ok ", "degraded ", "down "))]
    check((services, r.returncode), (["web-1", "cache-1"], 1))
    check((Path(ailleurs) / "reports").is_dir(), True)

## Critère final

L'aide de la commande installée annonce les cinq options :

```text
usage: fleetcheck [-h] [--inventory INVENTORY] [--env ENV]
                  [--format {text,json}] [--config CONFIG] [-v]
```

In [ ]:
with tempfile.TemporaryDirectory() as ailleurs:
    r = fleetcheck("--help", cwd=ailleurs)
check(all(option in r.stdout for option in ("--inventory", "--env", "--format", "--config", "--verbose")), True)

## Pour aller plus loin

- `python -m pip install build`, puis `python -m build` à la racine : `dist/` contient une wheel `fleetcheck-0.12.0-py3-none-any.whl`, installable par `pip install` sans le dépôt. Ajoutez `dist/` au `.gitignore`.
- Renommez `main` en `entry` dans `cli.py` sans toucher `pyproject.toml`, lancez `fleetcheck --help` dans un terminal, lisez l'`ImportError`, puis remettez `main`.